Helper Functions

In [ ]:
# from pathlib import Path

# def clear_cached_labels(directory_path="."):
#     """
#     Finds and deletes all files ending with '_labels.csv' in the given directory.
#     Change directory_path to your specific data folder if they aren't in the root.
#     """
#     # Create a Path object for the target directory
#     target_dir = Path(directory_path)
    
#     # Find all matching files
#     label_files = list(target_dir.glob("*_grid.csv"))
    
#     if not label_files:
#         print("No label files found. You are ready to generate new ones!")
#         return

#     print(f"Found {len(label_files)} label files. Deleting...")
    
#     # Iterate and delete
#     for file_path in label_files:
#         file_path.unlink()  # This physically deletes the file
#         print(f"Deleted: {file_path.name}")
        
#     print("Cleanup complete! Cache is clear.")

# # Run the function (adjust the path if your CSVs are in a specific folder like 'data/')
# clear_cached_labels("/kaggle/working")

In [ ]:
from pathlib import Path

INPUT_BASE = Path("/kaggle/input/notebooks/alecy333/glycemic-copy/")
if not INPUT_BASE.exists():
    INPUT_BASE = Path("/kaggle/input/datasets/alecy333/glycemia-processed")
    if not INPUT_BASE.exists():
        print("No notebook input detected... using working output directory.")
        INPUT_BASE = Path("/kaggle/working")


print(INPUT_BASE)

In [ ]:
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
warnings.filterwarnings("ignore")

BASE = Path(
    "/kaggle/input/datasets/akash01roy/glycemic-variability-and-wearable-device-data"
    "/big-ideas-lab-glycemic-variability-and-wearable-device-data-1.1.1"
)

food3_cols = ['date', 'time', 'time_begin', 'logged_food', 'amount',
       'unit', 'searched_food', 'calorie', 'total_carb', 'dietary_fiber',
       'sugar', 'protein', 'total_fat']

def to_relative_minutes(df, time_col="Timestamp"):
    df = df.copy()
    t0 = df[time_col].min()
    df["minutes"] = (df[time_col] - t0).dt.total_seconds() / 60.0
    return df


def load_dexcom(pid: str) -> pd.DataFrame:
    fpath = BASE / pid / f"Dexcom_{pid}.csv"
    df = pd.read_csv(fpath, low_memory=False)
    df = df.rename(columns={
        "Timestamp (YYYY-MM-DDThh:mm:ss)": "Timestamp",
        "Glucose Value (mg/dL)":           "Glucose",
        "Event Type":                       "EventType"
    })
    df = df[df["EventType"] == "EGV"].copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df["Glucose"]   = pd.to_numeric(df["Glucose"], errors="coerce")
    df = df.dropna(subset=["Timestamp", "Glucose"])
    df = df[["Timestamp", "Glucose"]].sort_values("Timestamp").reset_index(drop=True)
    df = to_relative_minutes(df)   # ← added
    return df


def load_wearable(pid: str, signal: str) -> pd.DataFrame:
    fpath = BASE / pid / f"{signal}_{pid}.csv"
    if not fpath.exists():
        print(f"  WARNING: {fpath.name} not found")
        return None
    df = pd.read_csv(fpath)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"datetime": "Timestamp"})
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)
    df = to_relative_minutes(df)   # ← added
    return df

def load_participant(pid: str) -> dict:
    data = {}
    data["Dexcom"] = load_dexcom(pid)
    for signal in ["EDA", "HR", "TEMP", "IBI", "ACC", "BVP"]:
        result = load_wearable(pid, signal)
        if result is not None:
            data[signal] = result
    
    # Food log — optional, don't crash if malformed
    food_path = BASE / pid / f"Food_Log_{pid}.csv"
    if food_path.exists():
        try:
            if pid == '003':
                food = pd.read_csv(food_path, names=food3_cols, header=None)
            else:            
                food = pd.read_csv(food_path)
            food.columns = food.columns.str.strip()  # clean column names
            
            # Try time_begin first, then time, then date+time
            if "time_begin" in food.columns:
                time_col = "time_begin"
            elif "time" in food.columns:
                time_col = "time"
            else:
                time_col = None
            
            if time_col:
                food[time_col] = pd.to_datetime(food[time_col], errors="coerce")
                t0 = data["Dexcom"]["Timestamp"].min()
                food["food_minutes"] = (
                    food[time_col] - t0
                ).dt.total_seconds() / 60.0
            
            data["Food"] = food
        except Exception as e:
            print(f"  Food log warning for {pid}: {e}")
    
    return data

In [ ]:
def resample_to_1min(df, value_col, mask_below=None):
    """
    Convert irregular signal into 1-min averaged time series.
    ONLY produces signal-level representation (no feature engineering).
    """
    df = df.copy()

    if mask_below is not None:
        df.loc[df[value_col] < mask_below, value_col] = np.nan

    df["minute_bin"] = df["minutes"].astype(int)

    # ONLY MEAN (important simplification)
    resampled = df.groupby("minute_bin")[value_col].mean()

    return resampled


def build_minute_grid(d):
    """
    Unified 1-minute aligned dataset.
    Each column = raw physiological signal (NOT engineered features).
    """
    max_min = int(d["Dexcom"]["minutes"].max())
    
    grid = pd.DataFrame({"minute": np.arange(0, max_min + 1)})
    grid = grid.set_index("minute")

    #time since midnight
    t0 = d["Dexcom"]["Timestamp"].min()
    start_minutes_from_midnight = t0.hour * 60 + t0.minute
    grid["minutes_from_midnight"] = (start_minutes_from_midnight + grid.index) % 1440

    # Glucose (interpolated only for continuity)
    gluc = (
        resample_to_1min(d["Dexcom"], "Glucose")
        .reindex(grid.index)
        .interpolate(method="linear", limit=15)
    )
    
    grid["glucose"] = gluc

    # Raw physiological signals (NO max/mean duplication here)
    grid["hr"]   = resample_to_1min(d["HR"], "hr")
    grid["eda"]  = resample_to_1min(d["EDA"], "eda")
    grid["temp"] = resample_to_1min(d["TEMP"], "temp", mask_below=25)

    # ACC magnitude
    acc = d["ACC"].copy()
    acc["magnitude"] = np.sqrt(
        acc["acc_x"]**2 + acc["acc_y"]**2 + acc["acc_z"]**2
    )
    grid["acc"] = resample_to_1min(acc, "magnitude")

    #MEAL RELATED FEATURES
    grid["is_postprandial"] = 0
    grid["time_since_meal"] = 999.0
    grid["meal_event"] = 0
    grid["carbs"] = 0.0
    grid["sugar"] = 0.0
    grid["protein"] = 0.0
    grid["calories"] = 0.0
    
    if "Food" in d and not d["Food"].empty:
        food_df = d["Food"]
        
        # Iterate through the actual food log rows
        for _, row in food_df.iterrows():
            if pd.notna(row.get("food_minutes")):
                m = int(row["food_minutes"])
                
                # Make sure the meal happened within our grid timeframe
                if m in grid.index:
                    # 1. Discrete Meal Indicators
                    grid.loc[m, "meal_event"] = 1
                    
                    # Add macros (using += in case they logged 2 things in the exact same minute)
                    # CHANGE "carbs", "sugar", etc., to match the exact CSV column names!
                    grid.loc[m, "carbs"] += row.get("total_carb", 0) if pd.notna(row.get("total_carb")) else 0
                    grid.loc[m, "sugar"] += row.get("sugar", 0) if pd.notna(row.get("sugar")) else 0
                    grid.loc[m, "protein"] += row.get("protein", 0) if pd.notna(row.get("protein")) else 0
                    grid.loc[m, "calories"] += row.get("calorie", 0) if pd.notna(row.get("calories")) else 0

        # 2. Continuous Context Variables (from our earlier discussion)
        meal_minutes = food_df["food_minutes"].dropna().sort_values().values
        for meal_time in meal_minutes:
            m = int(meal_time)
            window_end = min(m + 120, grid.index.max())
            grid.loc[m:window_end, "is_postprandial"] = 1
            mask = grid.index >= m
            grid.loc[mask, "time_since_meal"] = grid.index[mask] - m
        
    #prevents inflated numbers if they forgot to log meal. assumes glucose returns to fasting baseline after 3 hrs
    grid["time_since_meal"] = grid["time_since_meal"].clip(upper=180)
    
    return grid

#personalized labels
# def build_labels_only(grid, prediction_horizon=30, std_multiplier=1.0, window_mins=1440, min_baseline_mins=360, stride=5):
#     # 1. The Trailing Baseline (Strictly PAST data, no leakage)
#     rolling_mean = grid['glucose'].rolling(window=window_mins, min_periods=min_baseline_mins).mean()
#     rolling_std  = grid['glucose'].rolling(window=window_mins, min_periods=min_baseline_mins).std()
    
#     dynamic_threshold = rolling_mean + (rolling_std * std_multiplier)
    
#     # 2. The Target Event (Strictly FUTURE data)
#     if prediction_horizon > 0:
#         window_size = int(prediction_horizon + 1)
#         forward_max = grid['glucose'].iloc[::-1].rolling(
#             window=window_size, 
#             min_periods=1
#         ).max().iloc[::-1]
#     else:
#         forward_max = grid['glucose']
        
#     # 3. Create the binary label
#     labels = (forward_max > dynamic_threshold).astype(int)
    
#     # 4. Clean up (Drop rows where we are still building the initial baseline)
#     valid_mask = labels.notna() & grid['glucose'].notna() & dynamic_threshold.notna()
    
#     records = pd.DataFrame({
#         "minute": grid.index[valid_mask],
#         "label": labels[valid_mask].values,
#         "current_glucose": grid.loc[valid_mask, 'glucose'].values,
#         "future_max_glucose": forward_max[valid_mask].values,
#         "dynamic_threshold": dynamic_threshold[valid_mask].values
#     })
    
#     # 5. THE STRIDE: Apply the decimation right before returning
#     # iloc[::stride] simply takes every Nth row.
#     records = records.iloc[::stride].reset_index(drop=True)
    
#     return records

#hard labels
def build_labels_only(grid, prediction_horizon=30, rise_threshold=10, stride=5, min_future_points=3):
    """
    Scans the grid and returns a dataframe of labels mapped to the exact minute.
    """
    records = []
    max_min = grid.index.max()

    for t in range(0, max_min - prediction_horizon, stride):
        if t not in grid.index: continue
        
        glucose_now = grid.loc[t, "glucose"]
        if pd.isna(glucose_now): continue

        future = grid.loc[t : t + prediction_horizon, "glucose"].dropna()
        if len(future) < min_future_points: continue

        glucose_future_max = future.max()
        if pd.isna(glucose_future_max): continue

        rise = glucose_future_max - glucose_now
        
        records.append({
            "minute": t,
            "label": int(rise >= rise_threshold),
            "rise_mg_dl": rise
        })

    return pd.DataFrame(records)

In [ ]:
def build_feature_matrix(grid, labels_df, tau=0, physio_windows=[120], food_windows=[120, 480, 1440]):
    """
    1. Shifts and rolls the perfect 1-minute grid.
    2. Merges with the sparse labels_df.
    """
    # Create a copy so we don't modify the cached master grid
    df = grid.copy()
    out = pd.DataFrame(index=df.index)

    #add physiological features
    modalities = {"hr": "hr", "eda": "eda", "temp": "temp", "acc": "acc"}

    for w in physio_windows:
        for mod, col in modalities.items():
            lag = tau if isinstance(tau, int) else tau.get(mod, 0)
            
            # Because 'df' is exactly 1-min spaced, shift(lag) correctly shifts by 'lag' minutes
            shifted = df[col].shift(lag)
            
            # Rolling over window_size minutes

            tolerance = 0.8
            min_req = int(tolerance*w)
            r = shifted.rolling(window=w, min_periods=3)
    
            out[f"{mod}_mean_{w}m"] = r.mean()
            out[f"{mod}_std_{w}m"]  = r.std()
            out[f"{mod}_max_{w}m"]  = r.max()
            
            # Slope calculation
            out[f"{mod}_slope_{w}m"] = r.apply(
                lambda x: (x.iloc[-1] - x.iloc[0]) / (len(x) - 1) if len(x) >= 3 else np.nan,
                raw=False
            )

    blinding_lag = tau if isinstance(tau, int) else min(tau.values()) #heuristic lag added to nonphysiological features to prevent data leakage

    #add food columns
    food_cols = ["meal_event", "carbs", "sugar", "protein", "calories"]
    for col in food_cols:
        if col in df.columns:
            shifted = df[col].shift(blinding_lag)
            
            for w in food_windows:
                # min_periods=1 is crucial here! 
                # If we used 3, a single meal event surrounded by zeros would yield NaN.
                r = shifted.rolling(w, min_periods=1)
                
                if col == "meal_event":
                    out[f"Eatcnt_{w}m"] = r.sum()
                else:
                    out[f"{col}_{w}m"] = r.sum()

    #contextual columns
    context_cols = ["minutes_from_midnight", "is_postprandial", "time_since_meal"]
    for c in context_cols:
        if c in df.columns:
            out[c] = df[c].shift(blinding_lag)
            
    out = out.reset_index() # Pulls 'minute' out of the index to a column

    # MERGE: Only keep rows where we have a calculated label from our stride=5 process
    final_df = pd.merge(labels_df, out, on="minute", how="inner")
    
    # Drop rows where feature generation failed (e.g., due to the lag/window pushing into NaNs)
    feature_cols = [c for c in final_df.columns if c not in ["minute", "label", "rise_mg_dl"]]
    final_df = final_df.dropna(how="all", subset=feature_cols)

    return final_df

In [ ]:
def create_master_data(prediction_horizon, exclude=[]):
    ALL_PIDS = [f"{i:03d}" for i in range(1, 17)]
    master_data = {} 
    
    for pid in ALL_PIDS:
        if pid in exclude:
            continue
        
        print(f"Processing {pid}...", end=" ")
        try:
            # 1. Load raw data and build the continuous 1-minute baseline
            grid_path = INPUT_BASE / f"{pid}_grid.csv"
            
            if grid_path.exists():
                grid = pd.read_csv(grid_path, index_col="minute")
            else:
                dp   = load_participant(pid)
                grid = build_minute_grid(dp)
                grid.to_csv(f"{pid}_grid.csv")
    
            # 2. Build ONLY the sparse targets (stride=5)
            # FIX: Dynamically name the cache file based on the horizon to prevent cross-contamination
            label_filename = f"{pid}_labels_h{prediction_horizon}.csv"
            labels_path = INPUT_BASE / label_filename
            
            if labels_path.exists():
                labels_df = pd.read_csv(labels_path)
            else:
                labels_df = build_labels_only(grid, prediction_horizon=prediction_horizon)
                labels_df["pid"] = pid
                # Write to current working directory using the specific horizon filename
                labels_df.to_csv(label_filename, index=False)
            
            # 3. Store BOTH pieces in the master cache
            master_data[pid] = {
                "grid": grid,
                "labels": labels_df
            }      
            
            print(f"{len(labels_df)} samples, {labels_df['label'].mean():.1%} positive")
            
        except Exception as e:
            print(f"FAILED — {e}")

    return master_data

In [ ]:
def build_full_dataset(master_data, tau=0, physio_windows=[120], food_windows=[120,480,1440], save_csv=True, filename=None, verbose=False):
    """
    Build feature-engineered dataset from master_data.

    Parameters
    ----------
    master_data : dict
        pid -> dictionary containing "grid" (continuous 1-min data) 
               and "labels" (sparse targets)

    tau : int or dict
        physiological lag
        tau could be a single int, or a dict of dicts

    window_size : int
        window length in minutes

    Returns
    -------
    full_dataset : pd.DataFrame
    """

    feature_tables = []

    for pid, data_dict in master_data.items():
        
        # 1. Unpack the decoupled grid and labels for this participant
        grid = data_dict["grid"]
        labels_df = data_dict["labels"]

        lags = tau if isinstance(tau, int) else tau.get(pid, {}) #if tau is a dict of pid:dict of lags, grab the corresponding lag dict.
        
        # 2. Pass both into the updated feature matrix builder
        feat_df = build_feature_matrix(
            grid=grid,
            labels_df=labels_df,
            tau=lags,
            physio_windows=physio_windows,
            food_windows=food_windows
        )

        idx = int(pid)

        patient_row = demo_df[demo_df["ID"] == idx]
        if not patient_row.empty:
            hba1c_val = patient_row["HbA1c"].values[0]
            gender_str = str(patient_row["Gender"].values[0]).strip().upper()
            
            feat_df["hba1c"] = hba1c_val
            feat_df["sex"] = 1 if gender_str == "MALE" else 0
        else:
            # Fallback if a patient is missing from the demographics file
            feat_df["hba1c"] = np.nan
            feat_df["sex"] = np.nan
        feat_df["pid"] = pid

        feature_tables.append(feat_df)

    full_dataset = pd.concat(
        feature_tables,
        ignore_index=True
    )

    if save_csv:
        if filename is None:
            if isinstance(tau, dict):
                filename = "full_dataset_modality_lag.csv"
            else:
                filename = f"full_dataset_tau_{tau}.csv"

        full_dataset.to_csv(filename, index=False)

    if verbose:
        print(f"\nTotal samples: {len(full_dataset):,}")
        print(
            f"Participants: "
            f"{full_dataset['pid'].nunique()}"
        )
        print(
            f"Overall positive: "
            f"{full_dataset['label'].mean():.1%}"
        )
    
        print(
            full_dataset.groupby("pid")["label"]
            .agg(["count", "mean"])
            .rename(
                columns={
                    "count": "samples",
                    "mean": "pos_rate"
                }
            )
            .round(3)
            .to_string()
        )

    return full_dataset

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

def run_loso(dataset, feature_cols, model, scale=True, verbose=False, smote=False):
    """
    dataset:
        concatenated feature dataframe
        already contains engineered features

    model:
        sklearn-style estimator supporting
        fit() and predict_proba()

    feature_cols:
        columns to use as predictors
    """

    all_pids = dataset["pid"].unique()

    y_true_all = []
    y_prob_all = []
    per_pid_auc = {}

    fold_importances = np.zeros(len(feature_cols))
    all_importances = []
    
    for test_pid in all_pids:
        if verbose:
            print(f"Leave one out -- test_pid for this iteration: {test_pid}")
        train = dataset[dataset["pid"] != test_pid]
        test  = dataset[dataset["pid"] == test_pid]

        train = train.dropna(subset=feature_cols)
        test  = test.dropna(subset=feature_cols)

        if verbose:
            print(f"Train set length: {len(train)}")
            print(f"Test set length: {len(test)}")
        
        if len(train) == 0 or len(test) == 0:
            print("no valid train or test set")
            print(f"length of train: {len(train)}")
            print(f"length of test: {len(train)}")
            continue

        X_train = train[feature_cols].values
        y_train = train["label"].values
        X_test = test[feature_cols].values
        y_test = test["label"].values

        if isinstance(model, XGBClassifier):
            ratio = (y_train == 0).sum() / (y_train == 1).sum()
            model.set_params(scale_pos_weight=ratio)

        if y_train.sum() >= 6 and smote:
            sm = SMOTE(random_state=42, k_neighbors=5)
            X_train, y_train = sm.fit_resample(X_train, y_train)
            if verbose:
                print(f"Smote used. New dataset size: {len(X_train)}")
                print(f"New positive ratio: {(y_train == 0).sum() / len(y_train)}")
                
        if scale:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        model.fit(X_train, y_train)

        all_importances.append(model.feature_importances_)
        
        y_prob = model.predict_proba(X_test)[:, 1]

        # if test_pid == "015":
        #     print(f"\n--- DEEP DIVE PATIENT 015 ---")
        #     print(f"Total Test Rows: {len(y_test)}")
        #     print(f"True Spikes (y_test sum): {y_test.sum()}")
        #     print(f"Predicted Probs for Spikes: {y_prob[y_test == 1]}")
        #     print("-----------------------------\n")

        y_true_all.extend(y_test)
        y_prob_all.extend(y_prob)

        if len(np.unique(y_test)) > 1:
            per_pid_auc[test_pid] = roc_auc_score(
                y_test,
                y_prob
            )
    
    return (
        np.array(y_true_all),
        np.array(y_prob_all),
        per_pid_auc,
        all_importances
    )

# Data Loading and Preprocessing

In [ ]:
demo_df = pd.read_csv("/kaggle/input/datasets/alecy333/big-ideas-demographics-csv-fixed/Demographics.csv")
demo_df = demo_df.sort_values(by="ID")
demo_df.reset_index(drop=True)

In [ ]:
#build feature columns
FEATURE_COLS = []
FEATURE_COLS.extend(["sex", "hba1c"])
context_cols = ["minutes_from_midnight", "is_postprandial", "time_since_meal"]
food_cols = ["meal_event", "carbs", "sugar", "protein", "calories"]

FEATURE_COLS.extend(context_cols)

physio_windows = [120, 360, 1440]
food_windows = [120, 480, 1440]

modalities = {"hr": "hr", "eda": "eda", "temp": "temp", "acc": "acc"}
#f"{mod}_mean_{w}m"
for w in physio_windows:
    for mod in modalities.keys():
        FEATURE_COLS.append(f"{mod}_mean_{w}m")
        FEATURE_COLS.append(f"{mod}_std_{w}m")
        FEATURE_COLS.append(f"{mod}_max_{w}m")
        FEATURE_COLS.append(f"{mod}_slope_{w}m")

for w in food_windows:
    for col in food_cols:
        if col == "meal_event":
            FEATURE_COLS.append(f"Eatcnt_{w}m")
        else:
            FEATURE_COLS.append(f"{col}_{w}m")
print(FEATURE_COLS)
print(f"Number of Features: {len(FEATURE_COLS)}")

In [ ]:
print("Creating the master dictionary...")
master_data = create_master_data(prediction_horizon=30, exclude=["015"]) #NO prediction horizon. predict EXACTLY at time t. 

# Experiment A -- ZERO LAG BASELINE

## Create the Zero Lag Baseline Dataframe

In [ ]:
print("Building dataset (tau = 0)...")
filename = "full_data_exp_a_h15.csv"
no_lag_full_path = INPUT_BASE / filename

if no_lag_full_path.exists():
    print("path exists, loading cached data...")
    full_data_no_lag = pd.read_csv(no_lag_full_path)
else:
    print("creating full labeled dataset...")
    full_data_no_lag = build_full_dataset(
        master_data,
        tau=0,
        physio_windows=physio_windows,
        food_windows=food_windows,
        filename=filename,
        verbose=True
    )

print(f"[INFO] Dataset shape: {full_data_no_lag.shape}")
print(f"[INFO] Positive rate: {full_data_no_lag['label'].mean():.3f}")
print(f"[INFO] Participants: {full_data_no_lag['pid'].nunique()}")

missing = [c for c in FEATURE_COLS if c not in full_data_no_lag.columns]
print(f"\n[CHECK] Missing features: {missing}")

full_data_no_lag

In [ ]:
def perform_analysis(model, trial_name=""):
    print("\n Running LOSO-CV...")

    y_true_all, y_prob_all, per_pid_auc, all_importances = run_loso(
        full_data_no_lag,
        FEATURE_COLS_RF,
        model,
        verbose=True,
        scale=True
    )
    
    y_pred = (y_prob_all >= 0.5).astype(int)
    
    print(f"\n================ {trial_name} RESULTS ================")
    print(f"AUROC (overall): {roc_auc_score(y_true_all, y_prob_all):.3f}")
    
    print("\nPer-participant AUROC:")
    for pid, auc in per_pid_auc.items():
        print(f"  {pid}: {auc:.3f}")
    
    print(f"\nMean AUROC: {np.mean(list(per_pid_auc.values())):.3f}")
    print(f"Std AUROC:  {np.std(list(per_pid_auc.values())):.3f}")

    y_true_all, y_prob_all, per_pid_auc, all_importances, y_pred

## Random Forest Test

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

print("\n==============================")
print("EXPERIMENT A.1 — RANDOM FOREST ZERO LAG MODEL")
print("==============================")

# ─────────────────────────────────────────────
# Model
# ─────────────────────────────────────────────
rfmodel = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ─────────────────────────────────────────────
# LOSO evaluation
# ─────────────────────────────────────────────
print("\n Running LOSO-CV...")
y_true_all, y_prob_all, per_pid_auc, all_importances = run_loso(
    full_data_no_lag,
    FEATURE_COLS,
    rfmodel,
    scale=True,
    verbose=True,
    smote=False
)

# ─────────────────────────────────────────────
# Results
# ─────────────────────────────────────────────

#y_pred = (y_prob_all >= 0.5).astype(int)

print("\n================ A.1 RESULTS ================")
print(f"AUROC (overall): {roc_auc_score(y_true_all, y_prob_all):.3f}")

print("\nPer-participant AUROC:")
for pid, auc in per_pid_auc.items():
    print(f"  {pid}: {auc:.3f}")

print(f"\nMean AUROC: {np.mean(list(per_pid_auc.values())):.3f}")
print(f"Std AUROC:  {np.std(list(per_pid_auc.values())):.3f}")

## Feature Selection

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import math

all_pids = full_data_no_lag["pid"].unique()
avg_importances = np.sum(all_importances, axis=0)/len(all_pids)

# --- 1. IDENTIFY THE GLOBAL TOP FEATURES ---
# Create a dataframe of the aggregated importances
global_importance_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Global_Importance': avg_importances
}).sort_values(by='Global_Importance', ascending=False)

# Get the names and original indices of the Top 15 features
top_n = 15
top_features = global_importance_df['Feature'].head(top_n).tolist()
top_indices = [FEATURE_COLS.index(f) for f in top_features]


# --- 2. PLOT THE AGGREGATED (GLOBAL) IMPORTANCES ---
plt.figure(figsize=(10, 6))
sns.barplot(
    data=global_importance_df.head(top_n), 
    x='Global_Importance', 
    y='Feature', 
    palette='viridis'
)
plt.title(f"Aggregated Top {top_n} Feature Importances (Global Average)")
plt.xlabel("Mean Impurity Decrease")
plt.tight_layout()
plt.show()


# --- 3. PLOT THE INDIVIDUAL GRID (APPLES-TO-APPLES) ---
num_pids = len(all_pids)
cols = 4
rows = math.ceil(num_pids / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4), sharex=True)
axes = axes.flatten()

for i, (pid, importances) in enumerate(zip(all_pids, all_importances)):
    # Extract the scores for ONLY the global top 15 features
    individual_scores = [importances[idx] for idx in top_indices]
    
    sns.barplot(x=individual_scores, y=top_features, ax=axes[i], palette='viridis')
    axes[i].set_title(f"Participant: {pid}")
    axes[i].set_xlabel("Importance")
    
    # Hide Y-axis labels on the inner plots to keep the grid clean and tight
    if i % cols != 0:
        axes[i].set_ylabel("")
        axes[i].set_yticks([])

# Clean up any empty subplots if you have a number of pids that doesn't divide by 4 perfectly
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f"Individual Feature Reliance (Tracking the Global Top {top_n})", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#copied this from one of the below cells
useless_features = global_importance_df.tail(20)["Feature"].tolist()

FEATURE_COLS_RF = [f for f in FEATURE_COLS if f not in useless_features]
print(len(FEATURE_COLS_RF))

### Test Feature-Selected Forest

In [ ]:
print("\n==============================")
print("EXPERIMENT A.1.1 — RANDOM FOREST ZERO LAG MODEL, FEATURE-SELECTED")
print("==============================")

# ─────────────────────────────────────────────
# Model
# ─────────────────────────────────────────────
rfmodel = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ─────────────────────────────────────────────
# LOSO evaluation
# ─────────────────────────────────────────────
print("\n Running LOSO-CV...")

y_true_all, y_prob_all, per_pid_auc, all_importances = run_loso(
    full_data_no_lag,
    FEATURE_COLS_RF,
    rfmodel,
    verbose=True,
    scale=True
)

# ─────────────────────────────────────────────
# Results
# ─────────────────────────────────────────────

y_pred = (y_prob_all >= 0.5).astype(int)

print("\n================ A.1.1 RESULTS ================")
print(f"AUROC (overall): {roc_auc_score(y_true_all, y_prob_all):.3f}")

print("\nPer-participant AUROC:")
for pid, auc in per_pid_auc.items():
    print(f"  {pid}: {auc:.3f}")

print(f"\nMean AUROC: {np.mean(list(per_pid_auc.values())):.3f}")
print(f"Std AUROC:  {np.std(list(per_pid_auc.values())):.3f}")

## XGBoost Test

In [ ]:
print("\n==============================")
print("EXPERIMENT A.2 — XGBoost ZERO LAG MODEL")
print("==============================")

# ─────────────────────────────────────────────
# XGBoost Model
# ─────────────────────────────────────────────
xgbmodel = XGBClassifier(n_estimators=100,
                             random_state=42,
                             eval_metric="logloss",
                             verbosity=0)

# ─────────────────────────────────────────────
# LOSO evaluation
# ─────────────────────────────────────────────
print("\n Running LOSO-CV...")

y_true_all, y_prob_all, per_pid_auc, all_importances = run_loso(
    full_data_no_lag,
    FEATURE_COLS,
    xgbmodel,
    scale=True,
    verbose=True
)

# ─────────────────────────────────────────────
# Results
# ─────────────────────────────────────────────
y_pred = (y_prob_all >= 0.5).astype(int)

print("\n================ A.2 RESULTS ================")
print(f"AUROC (overall): {roc_auc_score(y_true_all, y_prob_all):.3f}")

print("\nPer-participant AUROC:")
for pid, auc in per_pid_auc.items():
    print(f"  {pid}: {auc:.3f}")

print(f"\nMean AUROC: {np.mean(list(per_pid_auc.values())):.3f}")
print(f"Std AUROC:  {np.std(list(per_pid_auc.values())):.3f}")

# Experiment B -- UNFORM LAG SWEEP

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score

def perform_uniform_sweep(model, lags=[5, 10, 15, 20, 25, 30, 45, 60]):
    print(f"{'Lag (min)':<12} {'AUROC':<8} {'F1-Spike':<10}")
    print("-" * 32)
    
    lag_results = {}
    per_pid_results = {}
    
    # We assume `master_data` and `FEATURE_COLS` are already loaded in memory
    for lag in lags:
        
        # 1. Build the feature dataset in memory using the cached grids
        lag_df = build_full_dataset(
            master_data, 
            tau=lag,
            physio_windows=physio_windows,
            food_windows=food_windows,
            save_csv=False # Set to True if you want to inspect each lag's CSV
        )
        
        # Safety check: if lag is so large it pushes all data to NaN
        if len(lag_df) == 0:
            print(f"{lag:<12} {'Failed: Empty dataset'}")
            continue
        
        # 2. Run LOSO-CV (ensure you pass the model and scale parameters)
        y_true, y_prob, per_pid_auc, _ = run_loso(
            dataset=lag_df, 
            feature_cols=FEATURE_COLS, 
            model=model, 
            scale=True,
            verbose=False
        )
        per_pid_results[lag] = per_pid_auc
        
        # 3. Evaluate
        if len(np.unique(y_true)) > 1:
            auc = roc_auc_score(y_true, y_prob)
            y_pred_lag = (y_prob >= 0.5).astype(int)
            f1 = f1_score(y_true, y_pred_lag)
            
            lag_results[lag] = {"auroc": auc, "f1": f1}
            
            # Suppress the massive build_full_dataset printouts during the sweep 
            # so you just get the clean table row:
            print(f"{lag:<12} {auc:<8.3f} {f1:<10.3f}")
        else:
            print(f"{lag:<12} {'Failed: Only one class present in predictions'}")
    
    # 4. Conclude
    if lag_results:
        best_lag = max(lag_results, key=lambda x: lag_results[x]["auroc"])
        print(f"\nBest uniform lag: {best_lag} minutes "
              f"(AUROC={lag_results[best_lag]['auroc']:.3f})")
    else:
        print("\nNo valid results obtained for any lag.")
    
    #check individual per_pid_results for each lag...
    if per_pid_results:
        print("="*60)
        for lag, per_pid_auc in sorted(per_pid_results.items()):
            print(f"\nPer-participant AUROC for {lag}:")
            for pid, auc in per_pid_auc.items():
                print(f"  {pid}: {auc:.3f}")
            print("="*60)
    else:
        print("no per pid results")

    reutrn lag_results, per_pid_results

## Random Forest Sweep

In [ ]:
print("\n==============================")
print("EXPERIMENT B.1 — UNIFORM LAG SWEEP RANDOM FOREST")
print("==============================\n")

rf_sweep_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_lag_res, rf_perpid_res = perform_uniform_sweep(rf_sweep_model)

## XGBoost Sweep

In [ ]:
print("\n==============================")
print("EXPERIMENT B.2 — UNIFORM LAG SWEEP XGBOOST")
print("==============================\n")

xgb_sweep_model = XGBClassifier(n_estimators=100,
                             random_state=42,
                             eval_metric="logloss",
                             verbosity=0)

xg_lag_res, xg_perpid_res = perform_uniform_sweep(xg_sweep_model)

# Experiment C -- Custom Lag

In [ ]:
import json

json_path = "/kaggle/input/notebooks/alecy333/wearable-signal-analysis/lags.json"
# Open the file and parse it directly into a dictionary
with open(json_path, 'r') as file:
    lag_dicts_all = json.load(file)

print(lag_dicts_all)
print(type(lag_dicts_all))

In [ ]:
print("\nbuilding dataset...")

full_data_varying_lag = build_full_dataset(
        master_data, 
        tau=lag_dicts_all,
        physio_windows=physio_windows,
        food_windows=food_windows,
        filename="full_dataset_experiment_c.csv",
        save_csv=True,
        verbose=True
    )
full_data_varying_lag

## Random Forest Custom Lag Test

In [ ]:
print("\n==============================")
print("EXPERIMENT C.1 — RANDOM FOREST ZERO LAG MODEL")
print("==============================")

rfmodel = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

y_true_all, y_prob_all, per_pid_auc, all_importances, y_pred = perform_analysis(rfmodel, "C.1")

## XGBoost Custom Lag Test

In [ ]:
y_true_all, y_prob_all, per_pid_auc, all_importances, y_pred = perform_analysis(xgbmodel, "C.2")